In [1]:
!pip install -r /kaggle/input/ey-water-quality-dataset/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 2.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of dask-expr to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install dask==2024.8.0 distributed==2024.8.0 odc-stac==0.3.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.7 MB/s eta 0:00:00
  Attempting uninstall: dask
    Found existing installation: dask 2024.10.0
    Uninstalling dask-2024.10.0:
      Successfully uninstalled dask-2024.10.0
  Attempting uninstall: distributed
    Found existing installation: distributed 2024.10.0
    Uninstalling distributed-2024.10.0:
      Successfully uninstalled distributed-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-expr 1.1.16 requires dask==2024.10.0, but you have dask 2024.8.0 which is incompatible.
rapids-dask-dependency 25.6.0 requires dask==2025.5.0, but you have dask 2024.8.0 which is incompatible.
rapids-dask-dependency 25.6.0 requires distributed==2025.5.0, but you have distributed 2024.8.0 which is incompatible.


In [3]:
import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from datetime import timedelta
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
from pyproj import Transformer

warnings.filterwarnings("ignore")

print("✓ All imports successful")

✓ All imports successful


In [4]:
STAC_API_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
COLLECTION = "landsat-c2-l2"

# Bands to extract
BANDS = ['blue', 'green', 'red', 'nir08', 'swir16', 'swir22', 'qa_pixel']
# lwir11 is thermal, useful for LST. qa_pixel for masking if needed

# Buffer Zones (meters)
ZONES = {
    '1km': (0, 1000),
    '5km_ring': (1000, 5000),
    '10km_ring': (5000, 10000)
}

# Processing Configuration
BATCH_SIZE = 100  # Samples per batch
MAX_WORKERS = min(os.cpu_count(), 4)  # Parallel workers

print(f"Configuration:")
print(f"  - Bands: {len(BANDS)}")
print(f"  - Zones: {list(ZONES.keys())}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Max workers: {MAX_WORKERS}")

Configuration:
  - Bands: 7
  - Zones: ['1km', '5km_ring', '10km_ring']
  - Batch size: 100
  - Max workers: 4


In [5]:

def compute_indices(ds):
    """
    Compute spectral indices from xarray dataset.
    ds must contain bands: blue, green, red, nir08, swir16, swir22.
    """
    eps = 1e-10  # Epsilon to avoid division by zero
    
    # Cast to float32 to avoid integer overflow and enable NaN
    ds = ds.astype('float32')
    
    # Apply Landsat C2 L2 scaling factors
    # Surface Reflectance: 0.0000275 * DN - 0.2
    sr_bands = ['blue', 'green', 'red', 'nir08', 'swir16', 'swir22']
    for b in sr_bands:
        if b in ds:
            ds[b] = ds[b].where(ds[b] != 0)  # Mask fill values
            ds[b] = ds[b] * 0.0000275 - 0.2
    
    # Thermal: 0.00341802 * DN + 149.0 (Kelvin)
    if 'lwir11' in ds:
        ds['lwir11'] = ds['lwir11'].where(ds['lwir11'] != 0)
        ds['lwir11'] = ds['lwir11'] * 0.00341802 + 149.0

    # Calculate spectral indices
    indices = {}
    
    # NDVI: Normalized Difference Vegetation Index
    indices['ndvi'] = (ds['nir08'] - ds['red']) / (ds['nir08'] + ds['red'] + eps)
    
    # NDWI: Normalized Difference Water Index
    indices['ndwi'] = (ds['green'] - ds['nir08']) / (ds['green'] + ds['nir08'] + eps)
    
    # NDBI: Normalized Difference Built-up Index
    indices['ndbi'] = (ds['swir16'] - ds['nir08']) / (ds['swir16'] + ds['nir08'] + eps)
    
    # MNDWI: Modified Normalized Difference Water Index
    indices['mndwi'] = (ds['green'] - ds['swir16']) / (ds['green'] + ds['swir16'] + eps)
    
    # EVI: Enhanced Vegetation Index
    indices['evi'] = 2.5 * ((ds['nir08'] - ds['red']) / 
                            (ds['nir08'] + 6 * ds['red'] - 7.5 * ds['blue'] + 1 + eps))
    
    # BSI: Bare Soil Index
    indices['bsi'] = ((ds['swir16'] + ds['red']) - (ds['nir08'] + ds['blue'])) / \
                     ((ds['swir16'] + ds['red']) + (ds['nir08'] + ds['blue']) + eps)
    
    return ds, indices

print("✓ Spectral indices function defined")


✓ Spectral indices function defined


In [6]:
# Cell 4: Single Point Processing Function (SUPER DEBUG VERSION)

def process_single_point(row, catalog):
    """
    Process a single sample point to extract Landsat features.
    """
    lat = row['Latitude']
    lon = row['Longitude']
    
    # Parse sample date
    try:
        sample_date = pd.to_datetime(row['Sample Date'], dayfirst=True)
        # print(f"    Parsed date: {sample_date}")
    except:
        sample_date = pd.to_datetime(row['Sample Date'])
        # print(f"    Parsed date (fallback): {sample_date}")

    # Search window: -30 days to sample date
    search_window_days = 30
    date_range = f"{(sample_date - timedelta(days=search_window_days)).strftime('%Y-%m-%d')}/{sample_date.strftime('%Y-%m-%d')}"
 #   print(f"    Date range: {date_range}")
    
    # Create 10km radius bounding box
    bbox_deg = 0.15
    bbox = [lon - bbox_deg, lat - bbox_deg, lon + bbox_deg, lat + bbox_deg]
  #  print(f"    BBox: {bbox}")

    # Search for Landsat scenes
   # print(f"    Searching STAC...")
    search = catalog.search(
        collections=[COLLECTION],
        bbox=bbox,
        datetime=date_range,
        query={"eo:cloud_cover": {"lt": 30}}
    )
    items = list(search.items())
    # print(f"    Found {len(items)} items with <10% cloud cover")

    # Fallback: expand search if no items found
    if not items:
      #  print(f"    Trying fallback: ±60 days, <20% cloud...")
        search_window_days = 60
        start_date = (sample_date - timedelta(days=60)).strftime('%Y-%m-%d')
        end_date = (sample_date + timedelta(days=60)).strftime('%Y-%m-%d')
        date_range = f"{start_date}/{end_date}"
        search = catalog.search(
            collections=[COLLECTION],
            bbox=bbox,
            datetime=date_range,
            query={"eo:cloud_cover": {"lt": 20}}
        )
        items = list(search.items())
       # print(f"    Fallback found {len(items)} items")
    
    if not items:
    #    print(f"    ✗ NO ITEMS FOUND - returning empty dict")
        return {}

    # Select best item
    items.sort(key=lambda x: abs(pd.to_datetime(x.properties['datetime']).tz_convert(None) - sample_date))
    best_item = items[0]
   # print(f"    Best item: {best_item.id}")
   # print(f"    Item date: {best_item.properties['datetime']}")
    
    # Load data using odc-stac
    # print(f"    Loading data with odc-stac...")
    try:
        data = stac_load(
            [best_item],
            bands=BANDS,
            crs="EPSG:3857",
            resolution=30,
            chunks={},
            bbox=bbox
        )
      #  print(f"    ✓ Data loaded successfully")
    except Exception as e:
       # print(f"    ✗ STAC LOAD FAILED: {e}")
        return {}

    if not data or 'time' not in data:
       # print(f"    ✗ Data is empty or missing time dimension")
        return {}
    
   # print(f"    Data dims: {data.dims}")
   # print(f"    Data vars: {list(data.data_vars)}")
         
    # Squeeze time dimension
    ds = data.isel(time=0)
    
    # Transform center point to EPSG:3857
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
    center_x, center_y = transformer.transform(lon, lat)
    
    # Calculate distance grid from center
    dist = np.sqrt((ds.x - center_x)**2 + (ds.y - center_y)**2)
    
    # Compute scaled bands and indices
    # print(f"    Computing indices...")
    ds_scaled, indices = compute_indices(ds)
    # print(f"    ✓ Computed {len(indices)} indices")
    
    # Extract statistics for each zone
    stats = {}
    feature_sets = {**{b: ds_scaled[b] for b in BANDS if b in ds_scaled}, **indices}
    
    stat_funcs = {
        'mean': np.nanmean,
        'median': np.nanmedian,
        'std': np.nanstd,
        'min': np.nanmin,
        'max': np.nanmax,
        'q25': lambda x: np.nanpercentile(x, 25),
        'q75': lambda x: np.nanpercentile(x, 75)
    }

    # Calculate statistics for each zone
    feature_count = 0
    for zone_name, (min_r, max_r) in ZONES.items():
        mask = (dist >= min_r) & (dist < max_r)
        
        if not mask.any():
            for name in feature_sets:
                for stat_name in stat_funcs:
                    stats[f"landsat_{name}_{zone_name}_{stat_name}"] = np.nan
            continue

        for name, arr in feature_sets.items():
            masked_data = arr.where(mask)
            vals = masked_data.values.flatten()
            vals = vals[~np.isnan(vals)]
            
            if len(vals) == 0:
                for stat_name in stat_funcs:
                    stats[f"landsat_{name}_{zone_name}_{stat_name}"] = np.nan
            else:
                for stat_name, func in stat_funcs.items():
                    try:
                        stats[f"landsat_{name}_{zone_name}_{stat_name}"] = func(vals)
                        feature_count += 1
                    except:
                        stats[f"landsat_{name}_{zone_name}_{stat_name}"] = np.nan
    
    # print(f"    ✓ Extracted {feature_count} feature values")
    
    # Add metadata
    stats['landsat_scene_id'] = best_item.id
    stats['landsat_date'] = pd.to_datetime(best_item.properties['datetime']).isoformat()
    
    # print(f"    ✓ TOTAL STATS KEYS: {len(stats)}")
    return stats

# print("✓ Super debug single point processing function defined")

In [7]:
# Cell 5: Batch Processing Function (DEBUG VERSION)

def process_batch(batch_df):
    """
    Process a DataFrame batch with detailed logging.
    """
    catalog = pystac_client.Client.open(STAC_API_URL, modifier=pc.sign_inplace)
    
    results = []
    for idx, row in batch_df.iterrows():
        try:
            print(f"\nProcessing: Lat={row['Latitude']}, Lon={row['Longitude']}, Date={row['Sample Date']}")
            row_stats = process_single_point(row, catalog)
            
            # DEBUG: Check if we got features
            if not row_stats or len(row_stats) < 5:
                print(f"  ⚠ WARNING: No features extracted! Got: {list(row_stats.keys())}")
            else:
                print(f"  ✓ Extracted {len(row_stats)} features")
            
            # Add identifiers
            row_stats['Latitude'] = row['Latitude']
            row_stats['Longitude'] = row['Longitude']
            row_stats['Sample Date'] = row['Sample Date']
            results.append(row_stats)
            
        except Exception as e:
            print(f"  ✗ ERROR: {e}")
            import traceback
            traceback.print_exc()
            
            res = {
                'Latitude': row['Latitude'], 
                'Longitude': row['Longitude'], 
                'Sample Date': row['Sample Date'],
                'error': str(e)
            }
            results.append(res)
            
    return pd.DataFrame(results)

print("✓ Debug batch processing function defined")

✓ Debug batch processing function defined


In [8]:
print("Loading datasets...")
train_df = pd.read_csv("/kaggle/input/ey-water-quality-dataset/water_quality_training_dataset.csv")
test_df = pd.read_csv("/kaggle/input/ey-water-quality-dataset/submission_template.csv")
# train_df = train_df.iloc[:5].copy()
print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print("\nTraining columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

Loading datasets...
Training samples: 9319
Test samples: 200

Training columns: ['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
Test columns: ['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']


In [9]:
name = 'train'
df = train_df

print(f"Processing {name} dataset ({len(df)} samples)...")

# Create batches
batches = [df[i:i + BATCH_SIZE] for i in range(0, len(df), BATCH_SIZE)]
print(f"Split into {len(batches)} batches")

all_results = []

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_batch, batch): batch for batch in batches}
    
    for future in tqdm(as_completed(futures), total=len(batches), desc=f"Extracting {name}"):
        try:
            res_df = future.result()
            all_results.append(res_df)
        except Exception as e:
            print(f"Batch failed: {e}")

if all_results:
    train_features = pd.concat(all_results, ignore_index=True)
    output_file = f"landsat_features_{name}.parquet"
    train_features.to_parquet(output_file, index=False)
    print(f"✓ Saved {output_file}")
    print(f"  Shape: {train_features.shape}")
    print(f"  Features extracted: {train_features.shape[1]}")
else:
    print(f"⚠ No results for {name}!")

Processing train dataset (9319 samples)...
Split into 94 batches


Extracting train:   0%|          | 0/94 [00:00<?, ?it/s]


Processing: Lat=-27.273889, Lon=28.49, Date=26-01-2011
Processing: Lat=-29.641944, Lon=30.6875, Date=23-03-2011

Processing: Lat=-22.769722, Lon=30.886944, Date=23-02-2011
Processing: Lat=-28.760833, Lon=17.730278, Date=02-01-2011


  ⚠ WARNING: No features extracted! Got: []

Processing: Lat=-27.00222, Lon=28.76528, Date=23-02-2011
  ✓ Extracted 275 features

Processing: Lat=-27.02314, Lon=28.59389, Date=23-02-2011
  ✓ Extracted 275 features

Processing: Lat=-27.044444, Lon=27.005, Date=26-01-2011
  ✓ Extracted 275 features

Processing: Lat=-26.861111, Lon=28.884722, Date=03-01-2011
  ✓ Extracted 275 features

Processing: Lat=-28.735556, Lon=29.820556, Date=23-03-2011
  ✓ Extracted 275 features

Processing: Lat=-26.648056, Lon=27.089444, Date=23-03-2011
  ✓ Extracted 275 features

Processing: Lat=-26.45, Lon=28.085833, Date=03-01-2011
  ✓ Extracted 275 features

Processing: Lat=-34.092222, Lon=21.295278, Date=26-01-2011
  ✓ Extracted 275 features

Processing: Lat=-27.38972222, Lon=26

Extracting train:   1%|          | 1/94 [22:41<35:10:12, 1361.43s/it]


Processing: Lat=-28.801278, Lon=31.955389, Date=13-04-2011
  ✓ Extracted 275 features

Processing: Lat=-25.99136, Lon=27.84208, Date=22-03-2011
  ✓ Extracted 275 features

Processing: Lat=-27.01011111, Lon=26.69808333, Date=22-02-2011
  ✓ Extracted 275 features

Processing: Lat=-27.398611, Lon=26.614722, Date=12-04-2011
  ✓ Extracted 275 features

Processing: Lat=-27.169444, Lon=29.233889, Date=22-02-2011
  ✓ Extracted 275 features

Processing: Lat=-30.398056, Lon=30.601944, Date=13-04-2011
  ✓ Extracted 275 features

Processing: Lat=-29.160278, Lon=26.573333, Date=12-04-2011
  ✓ Extracted 275 features

Processing: Lat=-26.861111, Lon=28.884722, Date=22-03-2011
  ✓ Extracted 275 features

Processing: Lat=-26.98472222, Lon=26.63227778, Date=22-02-2011
  ✓ Extracted 275 features

Processing: Lat=-30.568333, Lon=29.422778, Date=12-04-2011
  ✓ Extracted 275 features

Processing: Lat=-26.45, Lon=28.085833, Date=22-03-2011
  ✓ Extracted 275 features

Processing: Lat=-30.27402, Lon=30.69602,

...

  ✓ Extracted 275 features


Extracting train:  99%|█████████▉| 93/94 [10:21:12<04:31, 271.55s/it]

  ✓ Extracted 275 features

Processing: Lat=-27.01011111, Lon=26.69808333, Date=09-12-2015
  ✓ Extracted 275 features

Processing: Lat=-25.81048333, Lon=27.90955222, Date=09-12-2015
  ✓ Extracted 275 features

Processing: Lat=-24.670833, Lon=28.560833, Date=09-12-2015
  ✓ Extracted 275 features

Processing: Lat=-22.91028, Lon=29.61111, Date=09-12-2015
  ✓ Extracted 275 features

Processing: Lat=-28.747778, Lon=31.745833, Date=09-12-2015
  ✓ Extracted 275 features

Processing: Lat=-26.98472222, Lon=26.63227778, Date=09-12-2015
  ✓ Extracted 275 features

Processing: Lat=-27.671111, Lon=27.236944, Date=09-12-2015
  ✓ Extracted 275 features

Processing: Lat=-30.336111, Lon=27.359444, Date=09-12-2015
  ✓ Extracted 275 features

Processing: Lat=-34.092222, Lon=21.295278, Date=09-12-2015
  ✓ Extracted 275 features

Processing: Lat=-29.641944, Lon=30.6875, Date=10-12-2015
  ✓ Extracted 275 features

Processing: Lat=-22.769722, Lon=30.886944, Date=10-12-2015
  ✓ Extracted 275 features

Process

Extracting train: 100%|██████████| 94/94 [10:33:38<00:00, 404.46s/it]


✓ Saved landsat_features_train.parquet
  Shape: (9319, 279)
  Features extracted: 279


In [10]:
print(train_features.isna().sum())

landsat_blue_1km_mean      2454
landsat_blue_1km_median    2454
landsat_blue_1km_std       2454
landsat_blue_1km_min       2454
landsat_blue_1km_max       2454
                           ... 
landsat_date                 84
Latitude                      0
Longitude                     0
Sample Date                   0
error                      9311
Length: 279, dtype: int64


In [11]:
# Cell 8: Process Test Dataset
name = 'test'  # ← Different name
df = test_df   # ← Different dataframe (test_df instead of train_df)

print(f"Processing {name} dataset ({len(df)} samples)...")

# Create batches
batches = [df[i:i + BATCH_SIZE] for i in range(0, len(df), BATCH_SIZE)]
print(f"Split into {len(batches)} batches")

all_results = []

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_batch, batch): batch for batch in batches}
    
    for future in tqdm(as_completed(futures), total=len(batches), desc=f"Extracting {name}"):
        try:
            res_df = future.result()
            all_results.append(res_df)
        except Exception as e:
            print(f"Batch failed: {e}")

if all_results:
    test_features = pd.concat(all_results, ignore_index=True)  # ← Saves to test_features
    output_file = f"landsat_features_{name}.parquet"  # ← This becomes "landsat_features_test.parquet"
    test_features.to_parquet(output_file, index=False)
    print(f"✓ Saved {output_file}")
    print(f"  Shape: {test_features.shape}")
    print(f"  Features extracted: {test_features.shape[1]}")
else:
    print(f"⚠ No results for {name}!")

Processing test dataset (200 samples)...
Split into 2 batches


Extracting test:   0%|          | 0/2 [00:00<?, ?it/s]


Processing: Lat=-32.99163889, Lon=27.64002778, Date=30-05-2013

Processing: Lat=-32.043333, Lon=27.822778, Date=01-09-2014
  ✓ Extracted 275 features

Processing: Lat=-32.08639, Lon=25.57556, Date=09-04-2015
  ✓ Extracted 275 features

Processing: Lat=-33.329167, Lon=26.0775, Date=16-09-2015
  ✓ Extracted 275 features

Processing: Lat=-32.99163889, Lon=27.64002778, Date=07-05-2015
  ✓ Extracted 275 features

Processing: Lat=-33.731111, Lon=24.618333, Date=24-04-2014
  ✓ Extracted 275 features

Processing: Lat=-34.096389, Lon=24.439167, Date=07-02-2012
  ✓ Extracted 275 features

Processing: Lat=-32.00055556, Lon=28.58166667, Date=09-04-2014
  ✓ Extracted 275 features

Processing: Lat=-32.00055556, Lon=28.58166667, Date=01-10-2014
  ✓ Extracted 275 features

Processing: Lat=-32.08639, Lon=25.57556, Date=08-05-2015
  ✓ Extracted 275 features

Processing: Lat=-32.08639, Lon=25.57556, Date=19-07-2013
  ✓ Extracted 275 features

Processing: Lat=-32.605278, Lon=25.885, Date=08-11-2013
  ✓ E

Extracting test:  50%|█████     | 1/2 [31:44<31:44, 1904.59s/it]

  ✓ Extracted 275 features

Processing: Lat=-33.001667, Lon=25.161389, Date=05-02-2015
  ✓ Extracted 275 features

Processing: Lat=-32.605278, Lon=25.885, Date=15-07-2013
  ✓ Extracted 275 features

Processing: Lat=-32.043333, Lon=27.822778, Date=04-06-2012
  ✓ Extracted 275 features

Processing: Lat=-32.515278, Lon=28.015556, Date=13-01-2014
  ✓ Extracted 275 features


Extracting test: 100%|██████████| 2/2 [33:06<00:00, 993.36s/it]

✓ Saved landsat_features_test.parquet
  Shape: (200, 278)
  Features extracted: 278


In [12]:
# Load and examine the extracted features
train_features = pd.read_parquet("landsat_features_train.parquet")
test_features = pd.read_parquet("landsat_features_test.parquet")

print("Training Features Summary:")
print(f"  Shape: {train_features.shape}")
print(f"  Missing values: {train_features.isna().sum().sum()}")
print(f"\nSample columns:")
print(train_features.columns.tolist()[:20])

print("\n" + "="*80)
print("Test Features Summary:")
print(f"  Shape: {test_features.shape}")
print(f"  Missing values: {test_features.isna().sum().sum()}")

# Check for errors
if 'error' in train_features.columns:
    error_count = train_features['error'].notna().sum()
    print(f"\n⚠ Training errors: {error_count}/{len(train_features)}")
    
if 'error' in test_features.columns:
    error_count = test_features['error'].notna().sum()
    print(f"⚠ Test errors: {error_count}/{len(test_features)}")

# Display sample statistics
print("\nSample feature values (first row):")
feature_cols = [c for c in train_features.columns if c.startswith('landsat_')]
print(train_features[feature_cols[:5]].head(1))

Training Features Summary:
  Shape: (9319, 279)
  Missing values: 485675

Sample columns:
['landsat_blue_1km_mean', 'landsat_blue_1km_median', 'landsat_blue_1km_std', 'landsat_blue_1km_min', 'landsat_blue_1km_max', 'landsat_blue_1km_q25', 'landsat_blue_1km_q75', 'landsat_green_1km_mean', 'landsat_green_1km_median', 'landsat_green_1km_std', 'landsat_green_1km_min', 'landsat_green_1km_max', 'landsat_green_1km_q25', 'landsat_green_1km_q75', 'landsat_red_1km_mean', 'landsat_red_1km_median', 'landsat_red_1km_std', 'landsat_red_1km_min', 'landsat_red_1km_max', 'landsat_red_1km_q25']

Test Features Summary:
  Shape: (200, 278)
  Missing values: 6575

⚠ Training errors: 8/9319

Sample feature values (first row):
   landsat_blue_1km_mean  landsat_blue_1km_median  landsat_blue_1km_std  \
0               0.077828                 0.075468              0.023416   

   landsat_blue_1km_min  landsat_blue_1km_max  
0               0.03232              0.148645  
